# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## 01 · Repository Ingestion

This notebook demonstrates the **repository ingestion stage** of the RAG pipeline.

Given a public GitHub repository URL, the `GitHubRepositoryLoader` retrieves the repository files and converts them into a collection of LangChain `Document` objects.

The implementation of the ingestion stage is located in *src/github_rag/ingestion*:
* `repository.py`: Implements the `GitHubRepositoryLoader` class.
* `filters.py`: Defines the file types and files allowed for ingestion into the RAG pipeline.

#### **Notebook objectives**

* Load a GitHub repository using the `GitHubRepositoryLoader`.
* Explore the structure of the resulting `Document` objects:

  * Raw document content.
  * Metadata associated with each document.

## Imports and Configuration

In [3]:
from collections import Counter
import pandas as pd
from github_rag.ingestion.repository import GitHubRepositoryLoader

You can explore any GitHub repository simply by providing its URL. For this demo, I'll be using one of my own end-to-end Data Science projects that implements and evaluates a distributed Active Learning solution in PySpark.

In [4]:
REPOSITORY_URL = "https://github.com/davidfernxndez/spark-active-learning"

## 1. Repository Ingestion

This section demonstrates the basic behaviour of the `GitHubRepositoryLoader` by loading a public GitHub repository and inspecting the resulting LangChain `Document` objects.

In [5]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Number of documents loaded: {len(documents)}")

Number of documents loaded: 12


In [6]:
# Count the number of files for each type
file_types = Counter(
    document.metadata["file_type"]
    for document in documents
)

file_types_df = pd.DataFrame(
    file_types.items(),
    columns=["File type", "Documents"],
).sort_values(
    "Documents",
    ascending=False,
).reset_index(drop=True)

print("Document type distribution:")
file_types_df

Document type distribution:


,File type,Documents
0,.py,6
1,.ipynb,3
2,.md,2
3,.yml,1


## 2. Exploring LangChain Documents

The ingestion stage returns a list of LangChain `Document` objects.

Each `Document` contains two main components:
- `page_content`: the textual content of the file.
- `metadata`: information about the document's origin and type.

In [7]:
# Select a document to explore
document = documents[0]
print(type(document))

<class 'langchain_core.documents.base.Document'>


In [8]:
# Show the page content
print("Page content:")
print("-"*30)
print(document.page_content[:1000])
print("-"*30)

Page content:
------------------------------
name: spark-active-learning
channels:
  - conda-forge
  - defaults
  - pytorch
  - nvidia

dependencies:
  - python=3.8.10
  - ipython=7.25.0
  - ipywidgets=7.6.3
  - numpy=1.21.1
  - matplotlib=3.4.2
  - nb_conda_kernels=2.3.1
  - pandas=1.3.0
  - scipy=1.7.0
  - scikit-learn=1.0.2
  - jupyter=1.0.0
  - jupyterlab=3.0.16
  - graphviz=2.48.0
  - pydotplus=2.0.2
  - dtreeviz=1.3.1
  - pyspark=3.3.0
  - pip=21.1.3
  - shapely=1.8.5
  - owslib=0.27.2
  - rasterio=1.3.3
  - geopandas=0.12.1
  - contextily=1.3.0
  - tqdm=4.67.1
------------------------------


In [9]:
# Show the Metadata
print("\nMetadata:")
document.metadata


Metadata:


{'repository': 'davidfernxndez/spark-active-learning',
 'file_path': 'enviroment.yml',
 'file_type': '.yml',
 'source': 'https://github.com/davidfernxndez/spark-active-learning/blob/main/enviroment.yml',
 'branch': 'main'}

As we can see, the ingestion stage stores the following metadata:
- **repository**: repository name.
- **file_path**: path of the file within the repository.
- **file_type**: file extension.
- **source**: direct GitHub URL of the file.
- **branch**: checked-out repository branch.

This metadata will later be preserved throughout the RAG pipeline and used to identify the source of retrieved chunks.

We can inspect more documents:

In [10]:
for i, document in enumerate(documents[1:5]):
    print(f"--- Document {i} ---")
    print(f"File: {document.metadata['file_path']}")
    print(f"Type: {document.metadata['file_type']}")
    print(f"Source: {document.metadata['source']}")
    print(f"Content preview:\n{document.page_content[:300]}")
    print()

--- Document 0 ---
File: main.ipynb
Type: .ipynb
Source: https://github.com/davidfernxndez/spark-active-learning/blob/main/main.ipynb
Content preview:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "90ba9efb",
   "metadata": {},
   "source": [
    "# Active Learning with Apache Spark\n",
    "\n",
    "### Design and Evaluation of a Scalable Solution for Intelligent Instance Selection Based on Uncertainty and Diversity\n",
    "\n",
    "-

--- Document 1 ---
File: performance_experiment.ipynb
Type: .ipynb
Source: https://github.com/davidfernxndez/spark-active-learning/blob/main/performance_experiment.ipynb
Content preview:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "90442bfc",
   "metadata": {},
   "source": [
    "# Active Learning with Apache Spark\n",
    "\n",
    "### Design and Evaluation of a Scalable Solution for Intelligent Instance Selection Based on Uncertainty and Diversity\n",
    "\n",
    "-

--- Document 2 ---
File: README.md
Type: .md
Source: https